In [0]:
from pyspark.sql import functions as F, Window

df = spark.read.table("weather.bronze.mussoorie_1980_2026")
display(df.withColumn("row_num", F.row_number().over(Window.orderBy("time"))).filter("row_num % 1000 = 0").drop("row_num"))

In [0]:
display(df.dtypes)

Bronze Layer 
Loading the CSV/API response as-is with read_timestamp and _metadata columns.

In [0]:
path = f's3://weather-80-25/mussoorie_1980_2026.csv'

In [0]:
from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType, IntegerType
from pyspark.sql import functions as F

schema = StructType([
    StructField("time", TimestampType(), True),
    StructField("temperature_2m", DoubleType(), True),
    StructField("relative_humidity_2m", IntegerType(), True),
    StructField("cloud_cover_mid", IntegerType(), True),
    StructField("cloud_cover_high", IntegerType(), True),
    StructField("cloud_cover_low", IntegerType(), True),
    StructField("rain", DoubleType(), True),
    StructField("snowfall", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("dew_point_2m", DoubleType(), True),
    StructField("wind_speed_100m", DoubleType(), True),
    StructField("wind_speed_10m", DoubleType(), True),
    StructField("wind_direction_10m", IntegerType(), True),
])

# df = (
#     spark.read.format('csv')
#     .option('header', True)
#     .schema(schema)
#     .load(path)
#     .withColumn('read_timestamp', F.current_timestamp())
#     .select('*', '_metadata.file_name', '_metadata.file_size')
# )
# display(df.limit(10))

def weather_fetch(spark, path, schema):
    """
    Reads weather CSV data and appends metadata.
    """
    df = (
        spark.read.format('csv')
        .option('header', True)
        .schema(schema)
        .load(path)
        .withColumn('read_timestamp', F.current_timestamp())
        .select('*', '_metadata.file_name', '_metadata.file_size')
    )
    
    return df

my_weather_df = weather_fetch(spark, path, schema)
display(my_weather_df.limit(10))

Saving in memory data frame as permanent delta table 

In [0]:
%sql 
-- 1. Creating the main project catalog
CREATE CATALOG IF NOT EXISTS weather;

-- 2. Creating 3 schema inside the weather catalog
CREATE SCHEMA IF NOT EXISTS weather.bronze;
CREATE SCHEMA IF NOT EXISTS weather.silver;
CREATE SCHEMA IF NOT EXISTS weather.gold;



In [0]:
my_weather_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather.bronze.bronze_weather")

clearing extra leading 0 in timestamp

In [0]:
# df_clean = (
#     df
#     .withColumn(
#         "time",
#         F.to_timestamp(F.col("time"), "yyyy-MM-dd'T'HH:mm")
#     )
#     .withColumn(
#         "time",
#         F.date_format(F.col("time"), "yyyy-MM-dd HH:mm:ss")
#     )
# )

# display(df_clean.limit(10))


# We can't do this as this will change the schema of the dataframe from timestamp to string

In [0]:
# df_silver = df.withColumn(
#     "year", F.year("time")
# ).withColumn(
#     "month", F.month("time")
# ).withColumn(
#     "day", F.dayofmonth("time")
# ).withColumn(
#     "hour", F.hour("time")
# )

# display(df_silver.limit(20))

Checking if some of the columns are valid by checking 

- wind_direction_10m between 0 and 360
- relative_humidity_2m, cloud_cover_mid/high/low between 0 and 100
- temperature_2m within a realistic range for Mussoorie (roughly -30°C to 50°C)
- wind_speed_10m, wind_speed_100m non-negative and under a realistic max
- rain, precipitation, snowfall non-negative
- Cross-column check: dew_point_2m should never exceed temperature_2m

In [0]:
# df_silver = df_silver.withColumn(
#     "is_valid",
#     (F.col("wind_direction_10m").between(0, 360)) &
#     (F.col("relative_humidity_2m").between(0, 100)) &
#     (F.col("cloud_cover_mid").between(0, 100)) &
#     (F.col("cloud_cover_high").between(0, 100)) &
#     (F.col("cloud_cover_low").between(0, 100)) &
#     (F.col("temperature_2m").between(-30, 50)) &
#     (F.col("wind_speed_10m") >= 0) &
#     (F.col("wind_speed_100m") >= 0) &
#     (F.col("rain") >= 0) &
#     (F.col("precipitation") >= 0) &
#     (F.col("snowfall") >= 0) &
#     (F.col("dew_point_2m") <= F.col("temperature_2m"))
# )
# print(df_flagged.columns)


#No need to make a new column  

In [0]:

# def clean_silver_weather_data(df: DataFrame) -> DataFrame:
    
#     df = df.withColumn("date", to_date(col("time")))
    
#     df = df.withColumn(
#         "season",
#         when(month(col("time")).isin(12, 1, 2), "WR")
#         .when(month(col("time")).isin(3, 4, 5), "SG")
#         .when(month(col("time")).isin(6, 7, 8), "SR")
#         .when(month(col("time")).isin(9, 10, 11), "AU")
#     )
    
#     return df

In [0]:
# invalid_condition = ~(
#     (F.col("wind_direction_10m").between(0, 360)) &
#     (F.col("relative_humidity_2m").between(0, 101)) &
#     (F.col("cloud_cover_mid").between(0, 100)) &
#     (F.col("cloud_cover_high").between(0, 100)) &
#     (F.col("cloud_cover_low").between(0, 100)) &
#     (F.col("temperature_2m").between(-30, 50)) &
#     (F.col("wind_speed_10m") >= 0) &
#     (F.col("wind_speed_100m") >= 0) &
#     (F.col("rain") >= 0) &
#     (F.col("precipitation") >= 0) &
#     (F.col("snowfall") >= 0) 
#     # (F.col("dew_point_2m") <= F.col("temperature_2m"))
# )

# display(df_silver.filter(invalid_condition))

counting missing value in every column  

In [0]:
# from pyspark.sql.functions import col, sum

# # Count null values for each column and display the result
# missing_counts = df_silver.select([
#     sum(col(c).isNull().cast("int")).alias(c) for c in df_silver.columns
# ])

# display(missing_counts)

In [0]:
# # adding a decade partition
# from pyspark.sql.functions import floor
# df_silver = df_silver.withColumn(
#     "decade_partition", 
#     (floor(col("year") / 10) * 10).cast("int")
# )

**The Golden Layer**
The Gold layer is completely focused on the end-user or the specific business problem. Data here is usually heavily aggregated, summarized, and joined with other tables. You rarely look at individual hourly readings in the Gold layer.

In [0]:


# df_gold_daily = (
#     df_valid.groupBy("year", "month", "day")
#     .agg(
#         F.avg("temperature_2m").alias("avg_temp"),
#         F.min("temperature_2m").alias("min_temp"),
#         F.max("temperature_2m").alias("max_temp"),
#         F.sum("rain").alias("total_rain"),
#         F.sum("snowfall").alias("total_snowfall"),
#         F.avg("relative_humidity_2m").alias("avg_humidity"),
#         F.avg("wind_speed_10m").alias("avg_wind_speed"),
#         F.avg("cloud_cover_mid").alias("avg_cloud_cover_mid")
#     )
# )

In [0]:
# In Databricks, 'catalog.schema.weather_data_processing' refers to the table 'weather_data_processing'
# in the specified 'schema' within a particular 'catalog'. 'schema.weather_data_processing' refers to the
# table in the specified 'schema' within the current catalog. The catalog is a higher-level namespace,
# so omitting it uses the default catalog set for your session.